#### Memory Implementation

In [ ]:
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    query: str = Field(..., description="The search query for the tool")
    limit: int = Field(5, description="Number of results")


class CustomSearchTool(BaseTool):
    name: str = "Search Tool"
    description: str = "Searches the web for given query"
    args_schema: Type[BaseModel] = SearchInput  # Link Pydantic here

    def _run(self, query: str, limit: int) -> str:
        # Implementation using query and limit
        return f"Results for {query}: ..."


from crewai import Agent
researcher = Agent(
    role='Researcher',
    goal='Find info',
    backtrace='...',
    tools=[CustomSearchTool()]
)


In [ ]:
import json
from crewai import Agent, Task, Crew, Process, LLM, Memory
from crewai.tools import BaseTool, tool
from pydantic import BaseModel,TypeAdapter, Field
from typing import List, Dict
from com.example.agentic.tools.config import create_memory

#
memory = create_memory()

# 1. Define the individual object structure
class Item(BaseModel):
    id: int = Field(..., description="The id of the item")
    name: str = Field(..., description="The name of the item")
    price: float = Field(gt=0, description="The price of the item")
    quantity: int = Field(gt=0, description="the quantity of the item and quantity must be greater than zero")

# 2. Define the schema for the tool's arguments
class OrdersInput(BaseModel):
    orders: List[Item] = Field(description="Item list to process")

# Define a Pydantic model for the output structure
class AnalysisOutput(BaseModel):
    total_orders: int
    top_product: str
    total_quantity: int

# tool setup
@tool("Sum Function")
def my_simple_tool(question: str) -> str:
    """Tool to perform sum for json attribute."""
    _json = json.loads(question)
    print(_json)
    tool_output = f"enhanced_question {question} with clarifications"
    return tool_output

@tool("Order Analysis Tool")
def order_analysis_tool(orders: list) -> dict:
    """Tool to perform order analysis."""
    #_json = json.loads(orders)
    _total_orders=0
    _total_quantity = 0
    _top_product = 'Laptop'
    _ordersummery = {}
    if isinstance(orders, list):
        for _order in orders :
            _total_orders = _total_orders + 1
            _total_quantity = _total_quantity + _order['quantity']
    else:
        print("It's a JSON object (dictionary) or other type.")
    #
    _ordersummery['total_orders']=_total_orders
    _ordersummery['total_quantity']=_total_quantity
    _ordersummery['top_product']=_top_product
    return _ordersummery

# 3. Create the tool
class OrderAnalysisTool(BaseTool):
    name: str = "Order Analysis Tool"
    description: str = "Tool to perform order analysis."
    args_schema: type[BaseModel] = OrdersInput   # Link Pydantic here

    def _run(self, orders: List[Item]) -> str:
        print(orders)
        _total_quantity = 0
        _total_orders   = 0
        _top_product = 'Laptop'
        # Convert list of models to list of dicts
        #adapter = TypeAdapter(list[OrderInputModel])
        #orders_list = adapter.dump_python(orders.orders) 
        #print(orders_list)
        if isinstance(orders, list):
            _total_quantity = sum(_order.quantity for _order in orders)
            _total_orders   = len(orders)
            _top_product = 'Laptop'
        else:
            print("It's a JSON object (dictionary) or other type.")
        
        #
        _output = f"total_orders: {_total_orders} \ntop_product: {_top_product}\ntotal_quantity: {_total_quantity}"
        return _output
                
        

# Create an agent to collect orders
order_collector = Agent(
    role='Order Collector',
    goal='Collect the list of orders from provided input.',
    backstory="You are an expert in processing input order data.",
    verbose=True,
    tools=[],
    #llm=LLM(model="ollama/llama3.2:latest", base_url="http://localhost:11434")
)

# Create an agent to analyze orders
order_analyst = Agent(
    role='Order Analyst',
    goal='Analyze the list of orders and provide a oder summary report.',
    backstory="You are an expert in processing and analyzing order data.",
    verbose=True,
    tools=[OrderAnalysisTool()],
    memory=memory.scope("/agent/order_analyst"),
    #llm=LLM(model="ollama/llama3.2:latest", base_url="http://localhost:11434")
)

# Define the task using the Pydantic models for input and output validation
order_collector_task = Task(
    description="Analyze the incoming list of orders.",
    expected_output="provide a list of orders.",
    output_model=OrdersInput,  # Pydantic model for output structure
    agent=order_collector
)

order_analysis_task = Task(
    description="Analyze the incoming list of orders and provide a report on total orders, top product, and total quantity.",
    expected_output="provide a report on total orders, top product, and total quantity.",
    input_model=OrdersInput,  # Pydantic model for input validation
    output_model=AnalysisOutput,  # Pydantic model for output structure
    agent=order_analyst
)

# Define the crew and process
order_analysis_crew = Crew(
    agents=[order_collector],
    tasks=[order_collector_task],
    process=Process.sequential,
    verbose=True,
    #memory=memory,
    #llm=LLM(model="ollama/llama3.2:latest", base_url="http://localhost:11434")
)

#
from datetime import datetime
run_date = datetime.now().strftime('%Y-%m-%d')
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sample input data (list of orders)
input_data = {
    "orders":[
        {"id": 1, "name":"Laptop", "price":150.00, "quantity": 4},
        {"id": 2, "name":"Monitor", "price":100.5, "quantity": 5},
        {"id": 3, "name":"Laptop", "price":150.00, "quantity": 3},
        {"id": 4, "name":"Monitor", "price":100.5, "quantity": 4},
    ],
    "run_date":run_date,
    "run_id":run_id
}

# Kick off the task
print(f" order analysis crew triggered on {run_date} with execution id {run_id}")
result = order_analysis_crew.kickoff(inputs={'input_data': input_data})

# The result will be validated and structured by the output_model
print(result)

 order analysis crew triggered on 2026-04-27 with execution id 20260427_085435


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cd4fd9b6-69e6-429f-a9cd-e6f38b34ba22                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the incoming list of orders.                                                                     │
│  ID: e7266246-1ba7-4a41-870b-8eb1b972514f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 366.34ms                                                                                                 │
│  Content:                                                                                                       │
│  Relevant memories:                                                                                             │
│  - (score=0.74) Task: Analyze the incoming list of orders and provide a report on total orders, top product,    │
│  and total quantity.                                                                                            │
│  Agent: Order Analyst                                                                                           │
│  Expected result: provide a report on total orders, top product, and total quantity.                            │
│  Result: {"name": "order_analysis_tool", "parameters": {"orders":"[\"1\",""Product A)","\"20\","\"The quantity  │
│  30\"]"}}                                                                                                       │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: []                                                                                                   │
│  - (score=0.72) Task: Analyze the incoming list of orders and provide a report on total orders, top product,    │
│  and total quantity.                                                                                            │
│  Agent: Order Analyst                                                                                           │
│  Expected result: provide a report on total orders, top product, and total quantity.                            │
│  Result: """Analyze the incoming list of orders and retrieve relevant information."""                           │
│  {                                                                                                              │
│      "name": "save_to_memory",                                                                                  │
│      "parameters": {                                                                                            │
│          "contents": "[ Order ID, Product, Description, Quantity ]"                                             │
│      }                                                                                                          │
│  }                                                                                                              │
│                                                                                                                 │
│  {"name": "Search Memory", "parameters": {"queries": "['total orders', 'top product', 'quantity']"}}            │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: []                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze the incoming list of orders.                                                                     │
│  Agent: Order Collector                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: a6991c7f-4796-4e40-81ae-4514d6ace3aa                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/a6991c7f-4796-4e40-81ae-4514d6ace3aa?access_code=TRA │
│ CE-9f8dfd0183                                                                                                   │
│ 🔑 Access Code: TRACE-9f8dfd0183                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cd4fd9b6-69e6-429f-a9cd-e6f38b34ba22                                                                       │
│  Final Output: {"name": "Search Memory", "parameters": {"queries": "['total orders', 'top product',             │
│  'quantity']"}}                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{"name": "Search Memory", "parameters": {"queries": "['total orders', 'top product', 'quantity']"}}


╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 50842.70ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯